# **Setup and Imports**

In [2]:
!pip install -q "transformers>=4.40,<4.45"

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.7/43.7 kB 4.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.5/9.5 MB 163.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 48.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.6/3.6 MB 126.3 MB/s eta 0:00:00


In [3]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [4]:
import pandas
from transformers import (
    AutoTokenizer, AutoModelForSequenceClassification,
    TrainingArguments, Trainer, BatchEncoding,
    DataCollatorWithPadding, EvalPrediction,
    EarlyStoppingCallback
    )
from datasets import Dataset, DatasetDict, load_dataset, Value
import numpy as np
from scipy.stats import spearmanr

In [5]:
model_id = "MoritzLaurer/deberta-v3-large-zeroshot-v2.0"
tokenizer = AutoTokenizer.from_pretrained(model_id)
data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json: 0.00B [00:00, ?B/s]

spm.model:   0%|          | 0.00/2.46M [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

added_tokens.json:   0%|          | 0.00/23.0 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/970 [00:00<?, ?B/s]

In [6]:
def input_format(sample: pandas.Series) -> tuple[str, str]:
  story_part = [sample['precontext'], sample['sentence'], sample['ending'] or '']
  story_part = " ".join(story_part)
  meaning_part = (f"{sample['homonym']}: {sample['judged_meaning']} "
                   f"(e.g., \"{sample['example_sentence']}\")")

  return story_part, meaning_part


def tokenize_ambistory(batch: dict) -> BatchEncoding:
    story_parts = []
    meaning_parts = []
    for i in range(len(batch["precontext"])):
        ending = batch["ending"][i] or ''
        story = f"{batch['precontext'][i]} {batch['sentence'][i]} {ending}"
        meaning = (
            f"{batch['homonym'][i]}: {batch['judged_meaning'][i]} "
            f'(e.g., "{batch["example_sentence"][i]}")'
        )
        story_parts.append(story)
        meaning_parts.append(meaning)

    return tokenizer(
        story_parts,
        meaning_parts,
        truncation="only_first",
        max_length=256,
        padding=False,
    )

def tokenize_stsb(batch: dict) -> BatchEncoding:
    return tokenizer(
        batch["sentence1"],
        batch["sentence2"],
        truncation="only_first",
        max_length=256,
        padding=False
    )


def mark_target(sentence: str, start: int, end: int) -> str:
    """Wrap the target word with << >> markers (GlossBERT-style weak supervision)."""
    return sentence[:start] + "<<" + sentence[start:end] + ">>" + sentence[end:]


def tokenize_wic(batch: dict) -> BatchEncoding:
    """Tokenize WiC sentence pairs with target word marked in both sentences."""
    s1_marked, s2_marked = [], []
    for i in range(len(batch["sentence1"])):
        s1_marked.append(mark_target(batch["sentence1"][i], batch["start1"][i], batch["end1"][i]))
        s2_marked.append(mark_target(batch["sentence2"][i], batch["start2"][i], batch["end2"][i]))
    return tokenizer(
        s1_marked,
        s2_marked,
        truncation="only_first",
        max_length=128,
        padding=False,
    )


def metric_computation(eval_pred: EvalPrediction) -> dict[str, float]:
  predictions, labels = eval_pred
  predictions = predictions.squeeze()

  spearman_score, _ = spearmanr(predictions, labels)

  mean_absolute_error = np.mean(np.abs(predictions-labels))
  root_mean_squared_error = np.sqrt(np.mean((predictions-labels)**2))

  return {
      "spearman": spearman_score,
      "mae": mean_absolute_error,
      "rmse": root_mean_squared_error
  }


def wic_metric_computation(eval_pred: EvalPrediction) -> dict[str, float]:
    """WiC metrics: accuracy (primary, used for best-model selection) + spearman/mae/rmse.

    Labels are mapped to {1.0, 5.0}; we threshold predictions at the midpoint 3.0 to recover
    the binary same/different-sense decision."""
    predictions, labels = eval_pred
    predictions = predictions.squeeze()

    binary_labels = (labels >= 3.0).astype(int)
    binary_preds = (predictions >= 3.0).astype(int)
    accuracy = float(np.mean(binary_preds == binary_labels))

    spearman_score, _ = spearmanr(predictions, labels)
    mae = float(np.mean(np.abs(predictions - labels)))
    rmse = float(np.sqrt(np.mean((predictions - labels) ** 2)))

    return {
        "accuracy": accuracy,
        "spearman": spearman_score,
        "mae": mae,
        "rmse": rmse,
    }

# **Datasets**

In [7]:
ambistory_train_df = pandas.read_json("/content/drive/MyDrive/train.json").T
ambistory_validation_df = pandas.read_json("/content/drive/MyDrive/dev.json").T
ambistory_test_df = pandas.read_json("/content/drive/MyDrive/test.json").T

In [8]:
raw_datasets = DatasetDict({
    "ambistory_train_set": Dataset.from_pandas(ambistory_train_df),
    "ambistory_validation_set": Dataset.from_pandas(ambistory_validation_df),
    "ambistory_test_set": Dataset.from_pandas(ambistory_test_df)
})

for split_name in raw_datasets:
    raw_datasets[split_name] = raw_datasets[split_name].rename_column("average", "labels")

for split_name in raw_datasets:
    raw_datasets[split_name] = raw_datasets[split_name].cast_column("labels", Value("float32"))

print(raw_datasets["ambistory_train_set"].column_names)

cols_to_remove = [c for c in raw_datasets["ambistory_train_set"].column_names if c != "labels"]

tokenized_datasets = raw_datasets.map(
    tokenize_ambistory,
    batched=True,
    remove_columns=cols_to_remove,
)

Casting the dataset:   0%|          | 0/2280 [00:00<?, ? examples/s]

Casting the dataset:   0%|          | 0/588 [00:00<?, ? examples/s]

Casting the dataset:   0%|          | 0/930 [00:00<?, ? examples/s]

['homonym', 'judged_meaning', 'precontext', 'sentence', 'ending', 'choices', 'labels', 'stdev', 'nonsensical', 'sample_id', 'example_sentence', '__index_level_0__']


Map:   0%|          | 0/2280 [00:00<?, ? examples/s]

Map:   0%|          | 0/588 [00:00<?, ? examples/s]

Map:   0%|          | 0/930 [00:00<?, ? examples/s]

In [9]:
stsb = load_dataset("glue", "stsb")

stsb = stsb.rename_column("label", "labels")
stsb = stsb.map(lambda x: {"labels": float(x["labels"])})
stsb_tokenized = stsb.map(
    tokenize_stsb,
    batched=True,
    remove_columns=[c for c in stsb["train"].column_names if c != "labels"],
)

README.md: 0.00B [00:00, ?B/s]

stsb/train-00000-of-00001.parquet:   0%|          | 0.00/502k [00:00<?, ?B/s]

stsb/validation-00000-of-00001.parquet:   0%|          | 0.00/151k [00:00<?, ?B/s]

stsb/test-00000-of-00001.parquet:   0%|          | 0.00/114k [00:00<?, ?B/s]

Generating train split:   0%|          | 0/5749 [00:00<?, ? examples/s]

...

Map:   0%|          | 0/1500 [00:00<?, ? examples/s]

Map:   0%|          | 0/1379 [00:00<?, ? examples/s]

Map:   0%|          | 0/5749 [00:00<?, ? examples/s]

Map:   0%|          | 0/1500 [00:00<?, ? examples/s]

Map:   0%|          | 0/1379 [00:00<?, ? examples/s]

In [11]:
# WiC from SuperGLUE. Labels are binary: 0 = different sense, 1 = same sense.
# We map them to the same 1-5 scale used by STS-B and AmbiStory so the regression
# head trained in Stage 1 transfers cleanly. Mapping: 0 -> 1.0 (low plausibility),
# 1 -> 5.0 (high plausibility).

wic = load_dataset("super_glue", "wic")

# Test split has hidden labels (label = -1) per SuperGLUE convention. We don't use it.
del wic["test"]

LABEL_MAP = {0: 1.0, 1: 5.0}

wic = wic.map(lambda x: {"labels": LABEL_MAP[x["label"]]})

# Cast labels to float32 (same as AmbiStory/STS-B).
for split_name in wic:
    wic[split_name] = wic[split_name].cast_column("labels", Value("float32"))

wic_tokenized = wic.map(
    tokenize_wic,
    batched=True,
    remove_columns=[c for c in wic["train"].column_names if c != "labels"],
)

# Sanity check the test split has no usable labels (SuperGLUE convention: label = -1).
# We only train on train and evaluate on validation; the test split is unused.
print("WiC sizes:", {k: len(v) for k, v in wic_tokenized.items()})
print("Train label distribution:", {1.0: sum(1 for x in wic['train']['labels'] if x == 1.0),
                                     5.0: sum(1 for x in wic['train']['labels'] if x == 5.0)})

Casting the dataset:   0%|          | 0/5428 [00:00<?, ? examples/s]

Casting the dataset:   0%|          | 0/638 [00:00<?, ? examples/s]

Map:   0%|          | 0/5428 [00:00<?, ? examples/s]

Map:   0%|          | 0/638 [00:00<?, ? examples/s]

WiC sizes: {'train': 5428, 'validation': 638}
Train label distribution: {1.0: 2714, 5.0: 2714}


# **WiC Pretraining (single run)**

In [12]:
import gc
import torch

# ---------- Stage 1: STS-B (V2 config, 2 epochs - best STS-B config from prior ablation) ----------

model = AutoModelForSequenceClassification.from_pretrained(
    model_id,
    num_labels=1,
    problem_type="regression",
    ignore_mismatched_sizes=True,
)

stsb_args = TrainingArguments(
    output_dir="./curriculum_wic/stsb_seed42",
    eval_strategy="epoch",
    save_strategy="epoch",
    learning_rate=2e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=64,
    num_train_epochs=2,
    weight_decay=0.01,
    warmup_ratio=0.1,
    bf16=True,
    load_best_model_at_end=True,
    metric_for_best_model="spearman",
    greater_is_better=True,
    save_total_limit=1,
    logging_steps=200,
    report_to="none",
    seed=42,
)

stsb_trainer = Trainer(
    model=model,
    args=stsb_args,
    data_collator=data_collator,
    train_dataset=stsb_tokenized["train"],
    eval_dataset=stsb_tokenized["validation"],
    tokenizer=tokenizer,
    compute_metrics=metric_computation,
)

stsb_trainer.train()
print(f"Stage 1 (STS-B) dev Spearman: {stsb_trainer.state.best_metric:.4f}")
stsb_trainer.save_model("./curriculum_wic/stsb_seed42/final")

del model, stsb_trainer
gc.collect()
torch.cuda.empty_cache()

# ---------- Stage 2: WiC (regression head retained, labels mapped to {1.0, 5.0}) ----------

model = AutoModelForSequenceClassification.from_pretrained("./curriculum_wic/stsb_seed42/final")

wic_args = TrainingArguments(
    output_dir="./curriculum_wic/wic_seed42",
    eval_strategy="epoch",
    save_strategy="epoch",
    learning_rate=1e-5,                  # lower than STS-B to avoid forgetting the regression signal
    per_device_train_batch_size=16,
    per_device_eval_batch_size=64,
    num_train_epochs=3,
    weight_decay=0.01,
    warmup_ratio=0.1,
    bf16=True,
    load_best_model_at_end=True,
    metric_for_best_model="accuracy",    # report's protocol: select on WiC validation accuracy
    greater_is_better=True,
    save_total_limit=1,
    logging_steps=100,
    report_to="none",
    seed=42,
)

wic_trainer = Trainer(
    model=model,
    args=wic_args,
    data_collator=data_collator,
    train_dataset=wic_tokenized["train"],
    eval_dataset=wic_tokenized["validation"],
    tokenizer=tokenizer,
    compute_metrics=wic_metric_computation,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=2)],
)

wic_trainer.train()
print(f"Stage 2 (WiC) dev accuracy: {wic_trainer.state.best_metric:.4f}")
wic_trainer.save_model("./curriculum_wic/wic_seed42/final")

del model, wic_trainer
gc.collect()
torch.cuda.empty_cache()

# ---------- Stage 3: AmbiStory (same hyperparams as STS-B notebook) ----------

model = AutoModelForSequenceClassification.from_pretrained("./curriculum_wic/wic_seed42/final")

ambi_args = TrainingArguments(
    output_dir="./curriculum_wic/ambistory_seed42",
    eval_strategy="epoch",
    save_strategy="epoch",
    learning_rate=5e-6,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=64,
    num_train_epochs=5,
    weight_decay=0.01,
    warmup_ratio=0.06,
    bf16=True,
    load_best_model_at_end=True,
    metric_for_best_model="spearman",
    greater_is_better=True,
    save_total_limit=1,
    logging_steps=100,
    report_to="none",
    seed=42,
)

ambi_trainer = Trainer(
    model=model,
    args=ambi_args,
    data_collator=data_collator,
    train_dataset=tokenized_datasets["ambistory_train_set"],
    eval_dataset=tokenized_datasets["ambistory_validation_set"],
    tokenizer=tokenizer,
    compute_metrics=metric_computation,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=3)],
)

ambi_trainer.train()

test_metrics = ambi_trainer.evaluate(tokenized_datasets["ambistory_test_set"])
test_preds = ambi_trainer.predict(tokenized_datasets["ambistory_test_set"]).predictions.squeeze()
test_labels = ambistory_test_df["average"].to_numpy()
test_stdev = ambistory_test_df["stdev"].to_numpy()
threshold = np.maximum(test_stdev, 1.0)
acc_within_sd = float(np.mean(np.abs(test_preds - test_labels) <= threshold))

print(f"\n=== Curriculum (STS-B -> WiC -> AmbiStory), seed=42 ===")
print(f"  AmbiStory dev rho: {ambi_trainer.state.best_metric:.4f}")
print(f"  Test rho:          {test_metrics['eval_spearman']:.4f}")
print(f"  Test acc-SD:       {acc_within_sd:.4f}")
print(f"  Dev-test gap:      {ambi_trainer.state.best_metric - test_metrics['eval_spearman']:.4f}")
print(f"\nReference numbers (from STS-B notebook):")
print(f"  Baseline (AmbiStory only):           test rho=0.6336")
print(f"  STS-B v2 -> AmbiStory (seed=42):     test rho=0.6264")
print(f"  STS-B -> WiC -> AmbiStory (seed=42): test rho={test_metrics['eval_spearman']:.4f}")

config.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/870M [00:00<?, ?B/s]

Some weights of DebertaV2ForSequenceClassification were not initialized from the model checkpoint at MoritzLaurer/deberta-v3-large-zeroshot-v2.0 and are newly initialized because the shapes did not match:
- classifier.bias: found shape torch.Size([2]) in the checkpoint and torch.Size([1]) in the model instantiated
- classifier.weight: found shape torch.Size([2, 1024]) in the checkpoint and torch.Size([1, 1024]) in the model instantiated
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Epoch,Training Loss,Validation Loss,Spearman,Mae,Rmse
1,1.777100,0.416489,0.921553,0.506856,0.645360
2,0.196400,0.321298,0.927572,0.434638,0.566831


Stage 1 (STS-B) dev Spearman: 0.9276


Epoch,Training Loss,Validation Loss,Accuracy,Spearman,Mae,Rmse
1,2.817900,3.667186,0.691223,0.509992,1.416120,1.914990
2,1.660800,4.022805,0.711599,0.528353,1.318806,2.005693
3,0.955300,3.758716,0.741379,0.542548,1.151780,1.938741


Stage 2 (WiC) dev accuracy: 0.7414


Epoch,Training Loss,Validation Loss,Spearman,Mae,Rmse
1,1.102800,1.005577,0.620893,0.787877,1.002784
2,0.591200,0.948485,0.662410,0.763759,0.973902
3,0.396200,0.973123,0.663495,0.775416,0.986470
4,0.304300,0.935828,0.676043,0.756116,0.967382
5,0.242200,0.954821,0.679531,0.766481,0.977150



=== Curriculum (STS-B -> WiC -> AmbiStory), seed=42 ===
  AmbiStory dev rho: 0.6795
  Test rho:          0.6379
  Test acc-SD:       0.7548
  Dev-test gap:      0.0416

Reference numbers (from STS-B notebook):
  Baseline (AmbiStory only):           test rho=0.6336
  STS-B v2 -> AmbiStory (seed=42):     test rho=0.6264
  STS-B -> WiC -> AmbiStory (seed=42): test rho=0.6379


# **WiC Contribution Check Across Different Seeds**

In [13]:
import json
import gc
import torch
import numpy as np
from pathlib import Path

# ---------- locked curriculum config (matches STS-B notebook V2 + WiC stage) ----------

STSB_CONFIG = {
    "lr": 2e-5,
    "bs": 16,
    "epochs": 2,
    "warmup": 0.1,
}

WIC_CONFIG = {
    "lr": 1e-5,
    "bs": 16,
    "epochs": 3,
    "warmup": 0.1,
}

AMBISTORY_CONFIG = {
    "lr": 5e-6,
    "bs": 8,
    "epochs": 5,
    "warmup": 0.06,
}

SEEDS = [42, 1337, 2024]


def run_curriculum_wic(seed):
    """Run STS-B -> WiC -> AmbiStory pipeline with given seed.

    Returns the dev/test metrics for each stage."""

    # === STAGE 1: STS-B ===

    model = AutoModelForSequenceClassification.from_pretrained(
        model_id,
        num_labels=1,
        problem_type="regression",
        ignore_mismatched_sizes=True,
    )

    stsb_args = TrainingArguments(
        output_dir=f"./curriculum_wic_runs/stsb_seed{seed}",
        eval_strategy="epoch",
        save_strategy="epoch",
        learning_rate=STSB_CONFIG["lr"],
        per_device_train_batch_size=STSB_CONFIG["bs"],
        per_device_eval_batch_size=64,
        num_train_epochs=STSB_CONFIG["epochs"],
        weight_decay=0.01,
        warmup_ratio=STSB_CONFIG["warmup"],
        bf16=True,
        load_best_model_at_end=True,
        metric_for_best_model="spearman",
        greater_is_better=True,
        save_total_limit=1,
        logging_steps=200,
        report_to="none",
        seed=seed,
        disable_tqdm=True,
    )

    stsb_trainer = Trainer(
        model=model,
        args=stsb_args,
        data_collator=data_collator,
        train_dataset=stsb_tokenized["train"],
        eval_dataset=stsb_tokenized["validation"],
        tokenizer=tokenizer,
        compute_metrics=metric_computation,
    )

    stsb_trainer.train()
    stsb_dev = stsb_trainer.state.best_metric
    print(f"  Stage 1 (STS-B): dev Spearman = {stsb_dev:.4f}")

    stage1_ckpt = f"./curriculum_wic_runs/stsb_seed{seed}/final"
    stsb_trainer.save_model(stage1_ckpt)

    del model, stsb_trainer
    gc.collect()
    torch.cuda.empty_cache()

    # === STAGE 2: WiC ===

    model = AutoModelForSequenceClassification.from_pretrained(stage1_ckpt)

    wic_args = TrainingArguments(
        output_dir=f"./curriculum_wic_runs/wic_seed{seed}",
        eval_strategy="epoch",
        save_strategy="epoch",
        learning_rate=WIC_CONFIG["lr"],
        per_device_train_batch_size=WIC_CONFIG["bs"],
        per_device_eval_batch_size=64,
        num_train_epochs=WIC_CONFIG["epochs"],
        weight_decay=0.01,
        warmup_ratio=WIC_CONFIG["warmup"],
        bf16=True,
        load_best_model_at_end=True,
        metric_for_best_model="accuracy",
        greater_is_better=True,
        save_total_limit=1,
        logging_steps=200,
        report_to="none",
        seed=seed,
        disable_tqdm=True,
    )

    wic_trainer = Trainer(
        model=model,
        args=wic_args,
        data_collator=data_collator,
        train_dataset=wic_tokenized["train"],
        eval_dataset=wic_tokenized["validation"],
        tokenizer=tokenizer,
        compute_metrics=wic_metric_computation,
        callbacks=[EarlyStoppingCallback(early_stopping_patience=2)],
    )

    wic_trainer.train()
    wic_dev_acc = wic_trainer.state.best_metric
    print(f"  Stage 2 (WiC): dev accuracy = {wic_dev_acc:.4f}")

    stage2_ckpt = f"./curriculum_wic_runs/wic_seed{seed}/final"
    wic_trainer.save_model(stage2_ckpt)

    del model, wic_trainer
    gc.collect()
    torch.cuda.empty_cache()

    # === STAGE 3: AmbiStory ===

    model = AutoModelForSequenceClassification.from_pretrained(stage2_ckpt)

    ambi_args = TrainingArguments(
        output_dir=f"./curriculum_wic_runs/ambistory_seed{seed}",
        eval_strategy="epoch",
        save_strategy="epoch",
        learning_rate=AMBISTORY_CONFIG["lr"],
        per_device_train_batch_size=AMBISTORY_CONFIG["bs"],
        per_device_eval_batch_size=64,
        num_train_epochs=AMBISTORY_CONFIG["epochs"],
        weight_decay=0.01,
        warmup_ratio=AMBISTORY_CONFIG["warmup"],
        bf16=True,
        load_best_model_at_end=True,
        metric_for_best_model="spearman",
        greater_is_better=True,
        save_total_limit=1,
        logging_steps=100,
        report_to="none",
        seed=seed,
        disable_tqdm=True,
    )

    ambi_trainer = Trainer(
        model=model,
        args=ambi_args,
        data_collator=data_collator,
        train_dataset=tokenized_datasets["ambistory_train_set"],
        eval_dataset=tokenized_datasets["ambistory_validation_set"],
        tokenizer=tokenizer,
        compute_metrics=metric_computation,
        callbacks=[EarlyStoppingCallback(early_stopping_patience=3)],
    )

    ambi_trainer.train()

    ambi_dev = ambi_trainer.state.best_metric

    eval_logs = [
        log for log in ambi_trainer.state.log_history
        if "eval_spearman" in log and not np.isnan(log["eval_spearman"])
    ]
    best_log = max(eval_logs, key=lambda x: x["eval_spearman"])
    best_epoch = best_log["epoch"]

    test_metrics = ambi_trainer.evaluate(tokenized_datasets["ambistory_test_set"])

    test_preds = ambi_trainer.predict(tokenized_datasets["ambistory_test_set"]).predictions.squeeze()
    test_labels = ambistory_test_df["average"].to_numpy()
    test_stdev = ambistory_test_df["stdev"].to_numpy()
    threshold = np.maximum(test_stdev, 1.0)
    acc_within_sd = float(np.mean(np.abs(test_preds - test_labels) <= threshold))

    del model, ambi_trainer
    gc.collect()
    torch.cuda.empty_cache()

    return {
        "stsb_dev_spearman": float(stsb_dev),
        "wic_dev_accuracy": float(wic_dev_acc),
        "ambi_dev_spearman": float(ambi_dev),
        "ambi_best_epoch": float(best_epoch),
        "test_spearman": float(test_metrics["eval_spearman"]),
        "test_mae": float(test_metrics["eval_mae"]),
        "test_rmse": float(test_metrics["eval_rmse"]),
        "test_acc_within_sd": acc_within_sd,
        "dev_test_gap": float(ambi_dev - test_metrics["eval_spearman"]),
    }


Path("./curriculum_wic_runs").mkdir(exist_ok=True)
results = {}

for seed in SEEDS:
    print(f"\n{'='*60}")
    print(f"Seed {seed}")
    print('='*60)

    metrics = run_curriculum_wic(seed)
    results[seed] = metrics

    print(f"  -> STS-B dev:    {metrics['stsb_dev_spearman']:.4f}")
    print(f"  -> WiC dev acc:  {metrics['wic_dev_accuracy']:.4f}")
    print(f"  -> Ambi dev:     {metrics['ambi_dev_spearman']:.4f} (best epoch {metrics['ambi_best_epoch']:.0f})")
    print(f"  -> Test rho:     {metrics['test_spearman']:.4f}")
    print(f"  -> Test acc-SD:  {metrics['test_acc_within_sd']:.4f}")
    print(f"  -> Dev-test gap: {metrics['dev_test_gap']:.4f}")

    with open("./curriculum_wic_runs/multiseed_results.json", "w") as f:
        json.dump(results, f, indent=2)


print("\n\n" + "="*60)
print("MULTI-SEED CURRICULUM RESULTS (STS-B -> WiC -> AmbiStory)")
print("="*60)

test_spearmans = [r["test_spearman"] for r in results.values()]
test_acc_sds = [r["test_acc_within_sd"] for r in results.values()]
dev_spearmans = [r["ambi_dev_spearman"] for r in results.values()]
wic_accs = [r["wic_dev_accuracy"] for r in results.values()]
gaps = [r["dev_test_gap"] for r in results.values()]
best_epochs = [r["ambi_best_epoch"] for r in results.values()]

print(f"\nWiC dev accuracy:     {np.mean(wic_accs):.4f} +/- {np.std(wic_accs):.4f}")
print(f"Test Spearman:        {np.mean(test_spearmans):.4f} +/- {np.std(test_spearmans):.4f}")
print(f"Test Acc-within-SD:   {np.mean(test_acc_sds):.4f} +/- {np.std(test_acc_sds):.4f}")
print(f"AmbiStory dev rho:    {np.mean(dev_spearmans):.4f} +/- {np.std(dev_spearmans):.4f}")
print(f"Dev -> test gap:      {np.mean(gaps):.4f} +/- {np.std(gaps):.4f}")
print(f"Best epoch (Stage 3): {np.mean(best_epochs):.1f} (range: {min(best_epochs):.0f}-{max(best_epochs):.0f})")

print(f"\nPer seed:")
for seed, r in results.items():
    print(f"  seed={seed}: WiC acc={r['wic_dev_accuracy']:.4f}, test rho={r['test_spearman']:.4f}, "
          f"gap={r['dev_test_gap']:.4f}, best_epoch={r['ambi_best_epoch']:.0f}")

# Comparison vs prior runs from STS-B notebook
BASELINE_TEST_SPEARMAN = 0.6336        # AmbiStory-only single seed
STSB_V2_TEST_SPEARMAN  = 0.6264        # STS-B(v2) -> AmbiStory, seed=42

print(f"\nVs prior pipelines:")
print(f"  Delta vs AmbiStory-only baseline:   {np.mean(test_spearmans) - BASELINE_TEST_SPEARMAN:+.4f}")
print(f"  Delta vs STS-B(v2) -> AmbiStory:    {np.mean(test_spearmans) - STSB_V2_TEST_SPEARMAN:+.4f}")


Seed 42


Some weights of DebertaV2ForSequenceClassification were not initialized from the model checkpoint at MoritzLaurer/deberta-v3-large-zeroshot-v2.0 and are newly initialized because the shapes did not match:
- classifier.bias: found shape torch.Size([2]) in the checkpoint and torch.Size([1]) in the model instantiated
- classifier.weight: found shape torch.Size([2, 1024]) in the checkpoint and torch.Size([1, 1024]) in the model instantiated
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


{'loss': 1.8958, 'grad_norm': 15.97532844543457, 'learning_rate': 1.6049382716049385e-05, 'epoch': 0.5555555555555556}
{'eval_loss': 0.3933728039264679, 'eval_spearman': 0.9202009982772703, 'eval_mae': 0.48461776971817017, 'eval_rmse': 0.6271944046020508, 'eval_runtime': 1.5471, 'eval_samples_per_second': 969.58, 'eval_steps_per_second': 15.513, 'epoch': 1.0}
{'loss': 0.3311, 'grad_norm': 8.921812057495117, 'learning_rate': 9.876543209876543e-06, 'epoch': 1.1111111111111112}
{'loss': 0.2067, 'grad_norm': 5.182149887084961, 'learning_rate': 3.7037037037037037e-06, 'epoch': 1.6666666666666665}
{'eval_loss': 0.34026414155960083, 'eval_spearman': 0.9271880651646323, 'eval_mae': 0.45154869556427, 'eval_rmse': 0.5833216309547424, 'eval_runtime': 1.5722, 'eval_samples_per_second': 954.086, 'eval_steps_per_second': 15.265, 'epoch': 2.0}
{'train_runtime': 181.122, 'train_samples_per_second': 63.482, 'train_steps_per_second': 3.975, 'train_loss': 0.707232313685947, 'epoch': 2.0}
  Stage 1 (STS-B

Some weights of DebertaV2ForSequenceClassification were not initialized from the model checkpoint at MoritzLaurer/deberta-v3-large-zeroshot-v2.0 and are newly initialized because the shapes did not match:
- classifier.bias: found shape torch.Size([2]) in the checkpoint and torch.Size([1]) in the model instantiated
- classifier.weight: found shape torch.Size([2, 1024]) in the checkpoint and torch.Size([1, 1024]) in the model instantiated
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


{'loss': 1.8543, 'grad_norm': 7.075051307678223, 'learning_rate': 1.6049382716049385e-05, 'epoch': 0.5555555555555556}
{'eval_loss': 0.3495264947414398, 'eval_spearman': 0.9213428605380355, 'eval_mae': 0.45579564571380615, 'eval_rmse': 0.5912076830863953, 'eval_runtime': 1.5419, 'eval_samples_per_second': 972.833, 'eval_steps_per_second': 15.565, 'epoch': 1.0}
{'loss': 0.3188, 'grad_norm': 6.197659969329834, 'learning_rate': 9.876543209876543e-06, 'epoch': 1.1111111111111112}
{'loss': 0.2015, 'grad_norm': 4.5399699211120605, 'learning_rate': 3.7037037037037037e-06, 'epoch': 1.6666666666666665}
{'eval_loss': 0.35779106616973877, 'eval_spearman': 0.9274823446903007, 'eval_mae': 0.46388232707977295, 'eval_rmse': 0.5981563925743103, 'eval_runtime': 1.5883, 'eval_samples_per_second': 944.427, 'eval_steps_per_second': 15.111, 'epoch': 2.0}
{'train_runtime': 186.2145, 'train_samples_per_second': 61.746, 'train_steps_per_second': 3.867, 'train_loss': 0.6897187762790256, 'epoch': 2.0}
  Stage 1

Some weights of DebertaV2ForSequenceClassification were not initialized from the model checkpoint at MoritzLaurer/deberta-v3-large-zeroshot-v2.0 and are newly initialized because the shapes did not match:
- classifier.bias: found shape torch.Size([2]) in the checkpoint and torch.Size([1]) in the model instantiated
- classifier.weight: found shape torch.Size([2, 1024]) in the checkpoint and torch.Size([1, 1024]) in the model instantiated
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


{'loss': 1.5736, 'grad_norm': 16.039081573486328, 'learning_rate': 1.6049382716049385e-05, 'epoch': 0.5555555555555556}
{'eval_loss': 0.42447230219841003, 'eval_spearman': 0.9227039181675504, 'eval_mae': 0.5057134032249451, 'eval_rmse': 0.6515154242515564, 'eval_runtime': 1.5539, 'eval_samples_per_second': 965.319, 'eval_steps_per_second': 15.445, 'epoch': 1.0}
{'loss': 0.3157, 'grad_norm': 5.64858341217041, 'learning_rate': 9.876543209876543e-06, 'epoch': 1.1111111111111112}
{'loss': 0.2036, 'grad_norm': 4.780029773712158, 'learning_rate': 3.7037037037037037e-06, 'epoch': 1.6666666666666665}
{'eval_loss': 0.31038984656333923, 'eval_spearman': 0.9295115370876774, 'eval_mae': 0.42731449007987976, 'eval_rmse': 0.5571264028549194, 'eval_runtime': 1.566, 'eval_samples_per_second': 957.829, 'eval_steps_per_second': 15.325, 'epoch': 2.0}
{'train_runtime': 187.9961, 'train_samples_per_second': 61.161, 'train_steps_per_second': 3.83, 'train_loss': 0.6139927016364204, 'epoch': 2.0}
  Stage 1 (S

# **Ablation: WiC -> AmbiStory (no STS-B)**

Per the project plan: "+ WiC: Laurer -> WiC -> AmbiStory". This isolates WiC's
contribution without STS-B mixing in. Run this only after the multi-seed cell above
finishes if you want the full ablation table populated.

In [14]:
# Laurer -> WiC -> AmbiStory (no STS-B stage)

model = AutoModelForSequenceClassification.from_pretrained(
    model_id,
    num_labels=1,
    problem_type="regression",
    ignore_mismatched_sizes=True,
)

wic_only_args = TrainingArguments(
    output_dir="./wic_only/wic_seed42",
    eval_strategy="epoch",
    save_strategy="epoch",
    learning_rate=1e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=64,
    num_train_epochs=3,
    weight_decay=0.01,
    warmup_ratio=0.1,
    bf16=True,
    load_best_model_at_end=True,
    metric_for_best_model="accuracy",
    greater_is_better=True,
    save_total_limit=1,
    logging_steps=200,
    report_to="none",
    seed=42,
)

wic_only_trainer = Trainer(
    model=model,
    args=wic_only_args,
    data_collator=data_collator,
    train_dataset=wic_tokenized["train"],
    eval_dataset=wic_tokenized["validation"],
    tokenizer=tokenizer,
    compute_metrics=wic_metric_computation,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=2)],
)

wic_only_trainer.train()
print(f"WiC (no STS-B) dev accuracy: {wic_only_trainer.state.best_metric:.4f}")
wic_only_trainer.save_model("./wic_only/wic_seed42/final")

del model, wic_only_trainer
gc.collect()
torch.cuda.empty_cache()

model = AutoModelForSequenceClassification.from_pretrained("./wic_only/wic_seed42/final")

ambi_args_wo = TrainingArguments(
    output_dir="./wic_only/ambistory_seed42",
    eval_strategy="epoch",
    save_strategy="epoch",
    learning_rate=5e-6,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=64,
    num_train_epochs=5,
    weight_decay=0.01,
    warmup_ratio=0.06,
    bf16=True,
    load_best_model_at_end=True,
    metric_for_best_model="spearman",
    greater_is_better=True,
    save_total_limit=1,
    logging_steps=100,
    report_to="none",
    seed=42,
)

ambi_trainer_wo = Trainer(
    model=model,
    args=ambi_args_wo,
    data_collator=data_collator,
    train_dataset=tokenized_datasets["ambistory_train_set"],
    eval_dataset=tokenized_datasets["ambistory_validation_set"],
    tokenizer=tokenizer,
    compute_metrics=metric_computation,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=3)],
)

ambi_trainer_wo.train()

test_metrics_wo = ambi_trainer_wo.evaluate(tokenized_datasets["ambistory_test_set"])
test_preds_wo = ambi_trainer_wo.predict(tokenized_datasets["ambistory_test_set"]).predictions.squeeze()
test_labels = ambistory_test_df["average"].to_numpy()
test_stdev = ambistory_test_df["stdev"].to_numpy()
threshold = np.maximum(test_stdev, 1.0)
acc_within_sd_wo = float(np.mean(np.abs(test_preds_wo - test_labels) <= threshold))

print(f"\n=== WiC -> AmbiStory (no STS-B), seed=42 ===")
print(f"  Test rho:     {test_metrics_wo['eval_spearman']:.4f}")
print(f"  Test acc-SD:  {acc_within_sd_wo:.4f}")
print(f"  Dev-test gap: {ambi_trainer_wo.state.best_metric - test_metrics_wo['eval_spearman']:.4f}")

Some weights of DebertaV2ForSequenceClassification were not initialized from the model checkpoint at MoritzLaurer/deberta-v3-large-zeroshot-v2.0 and are newly initialized because the shapes did not match:
- classifier.bias: found shape torch.Size([2]) in the checkpoint and torch.Size([1]) in the model instantiated
- classifier.weight: found shape torch.Size([2, 1024]) in the checkpoint and torch.Size([1, 1024]) in the model instantiated
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Epoch,Training Loss,Validation Loss,Accuracy,Spearman,Mae,Rmse
1,4.737200,3.291630,0.697492,0.498198,1.460252,1.814285
2,1.906800,3.818907,0.710031,0.505559,1.362706,1.954202
3,0.834500,4.487387,0.716301,0.471792,1.340321,2.118345


WiC (no STS-B) dev accuracy: 0.7163


Epoch,Training Loss,Validation Loss,Spearman,Mae,Rmse
1,1.160900,1.213691,0.591812,0.885323,1.101676
2,0.667100,0.939467,0.647850,0.756405,0.969261
3,0.402500,1.025952,0.650410,0.796290,1.012893
4,0.351600,1.023403,0.651388,0.794951,1.011634
5,0.278200,0.986787,0.657206,0.777127,0.993372



=== WiC -> AmbiStory (no STS-B), seed=42 ===
  Test rho:     0.6384
  Test acc-SD:  0.7667
  Dev-test gap: 0.0188
